# Saudi Plate OCR — Train a Custom Recognizer

Fine-tunes the **Arabic PP-OCRv4 recognizer** (SVTR_LCNet) to read Saudi plates,
replacing EasyOCR. Reads the Arabic (top) and English (bottom) halves of each plate.

**Run cells top to bottom.** Set **Runtime → Change runtime type → GPU** first.
Stop after Section 6 (test). Section 7 (export) is only for later, when you're happy.


## 1. Install
Working stack for current Colab (Python 3.12): Paddle 2.6.2 + PaddleOCR 2.7 + NumPy<2.
Takes a few minutes.


In [ ]:
!apt-get -qq update && apt-get -qq -y install fonts-noto-core fonts-kacst fonts-hosny-amiri fonts-dejavu-core > /dev/null
!rm -rf PaddleOCR
!git clone -q --depth 1 -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q paddleocr 'paddle2onnx==1.1.0' onnxruntime
!pip install -q -r PaddleOCR/requirements.txt
!pip install -q 'numpy<2'   # imgaug + Paddle 2.6 need NumPy 1.x; installed last so it sticks
import paddle; print('paddle', paddle.__version__, '| GPU:', paddle.device.is_compiled_with_cuda())


## 2. Get our code from GitHub
Pulls `plate_spec.py`, `generate_plates.py`, `saudi_rec_config.yml`.

**One-time:** add a GitHub token to Colab Secrets (🔑 icon) named `GH_TOKEN` with repo read access.


In [ ]:
from google.colab import userdata
import subprocess
TOKEN = userdata.get('GH_TOKEN').strip()
subprocess.run('rm -rf saudi-plate-ocr', shell=True)
subprocess.run(['git','clone','-q',f'https://{TOKEN}@github.com/Tanzimalam1454/saudi-plate-ocr.git'])
for f in ['plate_spec.py','generate_plates.py','saudi_rec_config.yml']:
    subprocess.run(['cp', f'saudi-plate-ocr/{f}', '.'])
import os; assert all(os.path.exists(f) for f in ['plate_spec.py','generate_plates.py','saudi_rec_config.yml'])
print('files in place')


## 3. Generate synthetic data (multi-core)
8,000 plates -> 16,000 labeled half-crops, with a train/val split. ~5 min.
**Run once.** It writes `dataset/train_label.txt`, `dataset/val_label.txt`, `dataset/saudi_plate_dict.txt`.


In [ ]:
!rm -rf dataset
!python generate_plates.py --count 8000 --out dataset --montage
!echo '--- counts ---'; wc -l dataset/train_label.txt dataset/val_label.txt
from IPython.display import Image as IPImage; IPImage('dataset/_montage.jpg', width=900)


## 4. Download pretrained Arabic recognizer
The base model we fine-tune from. Its final layer is rebuilt for our character set.


In [ ]:
import os, glob
if not glob.glob('pretrain/arabic_PP-OCRv*_rec_train'):
    os.makedirs('pretrain', exist_ok=True)
    !cd pretrain && wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/multilingual/arabic_PP-OCRv4_rec_train.tar && tar xf arabic_PP-OCRv4_rec_train.tar
!ls pretrain


## 5. Train (saves the best model each epoch)
Fine-tunes for up to 12 epochs. After **every epoch** it evaluates on the val set and
saves `output/saudi_rec/best_accuracy` whenever val accuracy improves.

Watch the `best metric, acc:` line. **Stop the cell once it plateaus** — the best model is kept.


In [ ]:
import glob, re
pre = glob.glob('pretrain/arabic_PP-OCRv*_rec_train')[0] + '/best_accuracy'
n_train = sum(1 for l in open('dataset/train_label.txt') if l.strip())
iters = max(1, n_train // 64)                      # one epoch of iterations
cfg = open('saudi_rec_config.yml').read()
cfg = re.sub(r'pretrained_model:.*', f'pretrained_model: ./{pre}', cfg, count=1)
cfg = re.sub(r'eval_batch_step:.*', f'eval_batch_step: [0, {iters}]', cfg, count=1)
open('saudi_rec_config.yml','w').write(cfg)
print(f'{n_train} train crops -> eval + save best every {iters} iters (per epoch)')

!python PaddleOCR/tools/train.py -c saudi_rec_config.yml \
  -o Global.epoch_num=12 Optimizer.lr.warmup_epoch=1 \
     Train.loader.batch_size_per_card=64 Eval.loader.batch_size_per_card=64


## 6. Test on your own images (no export needed)
Upload **cropped plate images** (the whole plate, both rows). The cell splits each into
Arabic/English halves and reads them with your trained weights.

Files ending `__EN` = English reading, `__AR` = Arabic. (If your images are full scenes,
not plate close-ups, you need the plate detector first — ask and I'll add it.)


In [ ]:
from google.colab import files
from PIL import Image
import numpy as np, os
from plate_spec import split_saudi_plate

print('Upload cropped plate image(s):')
up = files.upload()
os.makedirs('myhalves', exist_ok=True)
for name in up:
    img = np.array(Image.open(name).convert('RGB'))[:, :, ::-1]   # -> BGR
    top, bottom = split_saudi_plate(img)
    stem = os.path.splitext(name)[0]
    Image.fromarray(bottom[:, :, ::-1]).save(f'myhalves/{stem}__EN.jpg')
    Image.fromarray(top[:, :, ::-1]).save(f'myhalves/{stem}__AR.jpg')
print('halves:', os.listdir('myhalves'))

!python PaddleOCR/tools/infer_rec.py -c saudi_rec_config.yml \
  -o Global.checkpoints=./output/saudi_rec/best_accuracy Global.infer_img=./myhalves


---
## 7. (LATER) Export to ONNX
Only run this once you're happy with the readings. Produces `inference/saudi_rec.onnx`
for the Jetson.


In [ ]:
!python PaddleOCR/tools/export_model.py -c saudi_rec_config.yml \
  -o Global.checkpoints=./output/saudi_rec/best_accuracy Global.save_inference_dir=./inference/saudi_rec
import os
mfile = 'inference.json' if os.path.exists('inference/saudi_rec/inference.json') else 'inference.pdmodel'
!paddle2onnx --model_dir ./inference/saudi_rec --model_filename {mfile} \
  --params_filename inference.pdiparams --save_file ./inference/saudi_rec.onnx --opset_version 13
!ls -la inference
